# Análise do Modelo com Lotes 1, 2 e 3 - Coral-sol 

Autor:  Viviane da Rosa Sommer

Atualizado: 01/11/2024

## Objetivo:
Notebook para avaliar o modelo treinado com os Lotes 1, 2 e 3 e com overlay, fazendo a inferência dos modelos de duto e coral-sol e analisando a intersecção das máscaras de resultado, realizando o cálculo de DICE e IoU.
As imagens utilizadas são da Prova de Fogo - PUC.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch

import glob
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os
import torchvision
import json
import torch
import csv
import zipfile
from typing import Callable, Any, Union
from IPython.display import display, Image
from PIL import Image as PILImage

## Declaração de Constantes e Modelos

In [ ]:
DUCT_CLASS_ID = 0
CORAL_CLASS_ID = 0
RESIZED_DIM_DUTO = 640
RESIZED_DIM_CORAL = 800

easy_duct_model = YOLO('runs/segment/train-facil-500-v1/weights/best.pt')
medium_duct_model = YOLO('runs/segment/train-medio-500-v1/weights/best.pt')
hard_duct_model = YOLO('runs/segment/train-dificil-500-v1/weights/best.pt')
coral_model = YOLO('runs/segment/train3/weights/best.pt')

## Funções Necessárias

In [ ]:
def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result with masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def generate_mask_from_polygon(polygon: list, height: int, width: int) -> np.ndarray:
    """
    Generates a binary mask from a specified polygon.

    Parameters:
        polygon (list): A list of (x, y) coordinates defining the vertices of the polygon.
        height (int): The height of the mask to be generated.
        width (int): The width of the mask to be generated.

    Returns:
        np.ndarray: A binary mask of size (height, width), where the polygon area is filled with 1.
    """
    mask = np.zeros((height, width), dtype=np.uint8)
    polygon = np.array(polygon, dtype=np.int32)  
    cv2.fillPoly(mask, [polygon], 1) 
    return mask


def zip_directories(directories: list, zip_file_name: str) -> None:
    """
    Compresses specified directories into a ZIP file.

    Parameters:
        directories (list): A list of directory paths to compress.
        zip_file_name (str): The name of the resulting ZIP file.

    Returns:
        None
    """
    with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for directory in directories:
            root_folder_name = os.path.basename(directory.rstrip('/\\'))  
            for root, _, files in os.walk(directory):
                for file in files:
                    full_path = os.path.join(root, file)
                    relative_path = os.path.relpath(full_path, directory)
                    unique_relative_path = os.path.join(root_folder_name, relative_path)
                    zipf.write(full_path, arcname=unique_relative_path)
                    
                    
def prediction_coral(img: np.array) -> tuple:
    """
    Faz uma previsão usando o modelo de coral e retorna a máscara para objetos de coral.

    Parameters:
        img (np.array): A imagem de entrada para previsão.

    Returns:
        tuple: Uma tupla contendo a máscara de coral redimensionada (np.ndarray) e sua versão tensor (torch.Tensor).
    """
    
    original_height, original_width = img.shape[:2]
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    
    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            coral_boxes = coral_boxes[nms_indices]
            coral_masks = coral_masks[nms_indices]
            final_coral_mask = torch.any(coral_masks, dim=0).int() * 255
        else:
            final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)

    final_coral_mask_np = final_coral_mask.cpu().numpy()
    mask_height, mask_width = final_coral_mask_np.shape

    resized_final_coral_mask = cv2.resize(final_coral_mask_np, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    centered_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    y_offset = (original_height - resized_final_coral_mask.shape[0]) // 2
    x_offset = (original_width - resized_final_coral_mask.shape[1]) // 2
    if (y_offset >= 0 and x_offset >= 0 and 
        y_offset + resized_final_coral_mask.shape[0] <= original_height and 
        x_offset + resized_final_coral_mask.shape[1] <= original_width):
        
        centered_mask[y_offset:y_offset + resized_final_coral_mask.shape[0], 
                      x_offset:x_offset + resized_final_coral_mask.shape[1]] = resized_final_coral_mask
    else:
        centered_mask = resized_final_coral_mask.copy()

    return centered_mask, torch.from_numpy(centered_mask).int()



def prediction_duct(img: np.array) -> tuple:
    """
    Makes a prediction using the duct model and returns the mask for duct objects.

    Parameters:
        img (np.array): The input image for prediction.

    Returns:
        tuple: A tuple containing the resized duct mask (np.ndarray) and its tensor version (torch.Tensor).
    """
    
    original_height, original_width = img.shape[:2]
    best_result = None
    for model in [easy_duct_model, medium_duct_model, hard_duct_model]:
        duct_results = model(img, verbose=False)
        best_result = process_results(duct_results)
        if best_result is not None:
            break

    if best_result is not None and best_result.masks is not None:
        masks = best_result.masks.data
        boxes = best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]  

        duct_indices = torch.where(classes == DUCT_CLASS_ID)[0]
        duct_boxes = boxes[duct_indices]
        duct_masks = masks[duct_indices]

        if len(duct_boxes) > 0:
            nms_indices = torchvision.ops.nms(duct_boxes[:, :4], scores[duct_indices], iou_threshold=0.2)
            duct_boxes = duct_boxes[nms_indices]
            duct_masks = duct_masks[nms_indices]
            final_duct_mask = torch.any(duct_masks, dim=0).int() * 255
        else:
            final_duct_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_duct_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
        
    final_duct_mask_np = final_duct_mask.cpu().numpy()
    resized_final_duct_mask = cv2.resize(final_duct_mask_np, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    return resized_final_duct_mask, torch.from_numpy(resized_final_duct_mask).int()


def read_annotation_file(filename: str) -> torch.Tensor:
    """
    Reads an annotation file and generates a mask for the coral object.

    Parameters:
        filename (str): The name of the file (image file format).

    Returns:
        torch.Tensor: A binary mask of the coral annotation.
    """
    if os.path.exists(filename.replace("jpg", "json")):
        with open(filename.replace("jpg", "json"), 'r') as f:
            annotation_data = json.load(f)
        annotated_coral_mask = torch.zeros((base_height, base_width), dtype=torch.uint8)
        for shape in annotation_data['shapes']:
            polygon = shape['points']
            mask = generate_mask_from_polygon(polygon, base_height, base_width)
            mask_tensor = torch.from_numpy(mask)
            annotated_coral_mask = annotated_coral_mask | mask_tensor
    else:
        annotated_coral_mask = torch.zeros((base_height, base_width), dtype=torch.uint8)
    annotated_coral_mask *= 255
    return annotated_coral_mask
                    
    
def generate_metrics(resized_final_coral_mask_tensor: torch.Tensor, 
                     resized_final_duct_mask_tensor: torch.Tensor, 
                     annotated_coral_mask: torch.Tensor) -> tuple:
    """
    Generates DICE and IoU metrics based on the provided masks.

    Parameters:
        resized_final_coral_mask_tensor (torch.Tensor): The resized coral mask tensor.
        resized_final_duct_mask_tensor (torch.Tensor): The resized duct mask tensor.
        annotated_coral_mask (torch.Tensor): The annotation coral mask.

    Returns:
        tuple: A tuple containing DICE, IoU, and the intersection mask (torch.Tensor).
    """
    intersection_mask = (resized_final_coral_mask_tensor & resized_final_duct_mask_tensor).int()
    intersection = torch.sum(intersection_mask & annotated_coral_mask)
    union = torch.sum(intersection_mask) + torch.sum(annotated_coral_mask)
    dice = (2.0 * intersection) / union if union > 0 else 1.0
    union_iou = torch.sum((intersection_mask | annotated_coral_mask).int())
    iou = intersection / union_iou if union_iou > 0 else 1.0
    return dice, iou, intersection_mask


def save_images(resized_final_duct_mask: np.ndarray, 
                resized_final_coral_mask: np.ndarray, 
                intersection_mask: np.ndarray, 
                img: np.ndarray, 
                final_directory: str,
                expert_annotation: torch.Tensor) -> None:
    filename_simple = filename.split("/")[-1].split(".")[0]

    plt.figure(figsize=(30, 21))
    
    plt.subplot(1, 6, 1)
    plt.title('Duct Mask')
    plt.imshow(resized_final_duct_mask, cmap='gray')
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    plt.subplot(1, 6, 2)
    plt.title('Coral Mask')
    plt.imshow(resized_final_coral_mask, cmap='gray')
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    intersection_mask = intersection_mask.numpy().astype(np.uint8)  
    plt.subplot(1, 6, 3)
    plt.title('Intersection')
    plt.imshow(intersection_mask, cmap='gray')
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    expert_mask = expert_annotation.numpy().astype(np.uint8)  
    img_copy = img.copy()
    if expert_mask.any():
        contours, _ = cv2.findContours(expert_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img_copy, contours, -1, (0, 255, 0), thickness=2) 

    plt.subplot(1, 6, 4)
    plt.title('Expert Annotation')
    plt.imshow(cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    model_img_copy = img.copy()
    if intersection_mask.any():  
        contours, _ = cv2.findContours(intersection_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(model_img_copy, contours, -1, (0, 0, 255), thickness=2)  

    plt.subplot(1, 6, 5)
    plt.title('Model Annotation')
    plt.imshow(cv2.cvtColor(model_img_copy, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    plt.subplot(1, 6, 6)
    plt.title('Original')
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.gca().set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.savefig(final_directory + f"/densidade_{filename_simple}.png")
    plt.show()
    plt.close()
    
    
def read_image(filename: str) -> tuple:
    """
    Reads an image from a file and returns it along with its height and width.

    Parameters:
        filename (str): The path to the image file.

    Returns:
        tuple: A tuple containing:
            - img (np.ndarray): The loaded image in BGR format.
            - base_height (int): The height of the image.
            - base_width (int): The width of the image.
            If the image cannot be loaded, returns (None, None, None).
    """
    img = cv2.imread(filename)
    if img is None:
        print(f"Failed to load image: {filename}")
        return None, None, None
        
    base_height, base_width = img.shape[:2]
    return img, base_height, base_width


def save_csv(filename: str, csv_filename: str, iou: Union[torch.Tensor, float], dice: Union[torch.Tensor, float]) -> None:
    """
    Appends the filename, folder name, IoU, and Dice values to a CSV file.

    Parameters:
        filename (str): The path to the image file.
        csv_filename (str): The path to the CSV file where the results will be saved.
        iou (Union[torch.Tensor, float]): The IoU metric value.
        dice (Union[torch.Tensor, float]): The Dice metric value.

    Returns:
        None
    """
    filename_simple = filename.split("/")[-1].split(".")[0]
    folder_filename = filename.split("/")[-2]
    iou_value = iou.item() if isinstance(iou, torch.Tensor) else iou
    dice_value = dice.item() if isinstance(dice, torch.Tensor) else dice
    with open(csv_filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([filename_simple, folder_filename, iou_value, dice_value])
        

## Processamento das imagens, salvando os resultados

In [ ]:
final_directory = "modelo_coral_lotes_1_2_3_com_overlay_resultados_1"
os.makedirs(final_directory, exist_ok=True)

csv_filename = os.path.join(final_directory, 'modelo_coral_lotes_1_2_3_com_overlay_resultados_1.csv')
results = []
with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Image Filename', 'Previous Folder', 'IoU', 'DICE'])

for i, filename in enumerate(glob.glob("puc_dataset/**/*.jpg")):
    
    img, base_height, base_width = read_image(filename)
    if img is None:
        continue

    resized_final_duct_mask, resized_final_duct_mask_tensor = prediction_duct(img)
    resized_final_coral_mask, resized_final_coral_mask_tensor = prediction_coral(img)
    
    annotated_coral_mask = read_annotation_file(filename)
    
    dice, iou, intersection_mask = generate_metrics(resized_final_coral_mask_tensor, resized_final_duct_mask_tensor, annotated_coral_mask)
    
    save_csv(filename, csv_filename, iou, dice)  
    save_images(resized_final_duct_mask, resized_final_coral_mask, intersection_mask, img, final_directory, annotated_coral_mask)

## Conversão do formato do notebook: .ipynb para .html

In [ ]:
!jupyter nbconvert --to html 'model_metrics.ipynb'

## Gerar zip para baixar, com os resultados obtidos

In [ ]:
directories_to_zip = [
    'modelo_coral_lotes_1_2_3_com_overlay_resultados_1',
]

zip_file_name = 'modelo_coral_lotes_1_2_3_com_overlay_resultados_1.zip'
zip_directories(directories_to_zip, zip_file_name)